In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import joblib

In [2]:
# Step - 1: Load dataset
from pathlib import Path
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

df = pd.read_csv(DATA_DIR/"credit_card_transactions.csv")
df.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud', 'merch_zipcode'],
      dtype='str')

In [3]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

df['age'] = ((df['trans_date_trans_time'] - df['dob']).dt.days / 365.25).astype(int)

df['month'] = df['trans_date_trans_time'].dt.month

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)


DROPPED = ['Unnamed: 0', 'trans_date_trans_time', 'cc_num',\
            'merchant', 'first', 'last',\
            'gender','street', 'city',\
            'zip','job', 'dob', 'unix_time','trans_num']

df = df.drop(columns=DROPPED)
df.columns

Index(['category', 'amt', 'state', 'lat', 'long', 'city_pop', 'merch_lat',
       'merch_long', 'is_fraud', 'merch_zipcode', 'age', 'month', 'month_sin',
       'month_cos'],
      dtype='str')

In [4]:
df = df.drop(columns=['merch_zipcode'])

In [5]:

numeric_cols = ['amt',
            'lat', 'long', 
            'city_pop', 'merch_lat', 
            'merch_long', 'age',
            'month_sin', 'month_cos']

catagorical_features = ['state', 'category']

preproccessing = ColumnTransformer(
    transformers = [
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown='ignore'), catagorical_features)
    ]
)

In [6]:
target = 'is_fraud'
features = ['category', 'amt', 'state',
             'lat', 'long', 'city_pop', 
             'merch_lat', 'merch_long', 'age', 
             'month_sin', 'month_cos']
print(df[target].value_counts())
df.columns

is_fraud
0    1289169
1       7506
Name: count, dtype: int64


Index(['category', 'amt', 'state', 'lat', 'long', 'city_pop', 'merch_lat',
       'merch_long', 'is_fraud', 'age', 'month', 'month_sin', 'month_cos'],
      dtype='str')

In [7]:
x = df[features]
y = df[target]

x_train, x_test, y_train, y_test = train_test_split(x, y, stratify = y, test_size = 0.2, random_state = 42)
print(f"Test Size {x_test.shape[0]}")
print(f"Train Size {x_train.shape[0]}")

Test Size 259335
Train Size 1037340


In [8]:
x_train_preprocessed = preproccessing.fit_transform(x_train)
x_test_preprocessed = preproccessing.transform(x_test)

In [9]:
# Using SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, Y_train_resampled = smote.fit_resample(x_train_preprocessed, y_train)

In [10]:
print(f"Resampled training set size {X_train_resampled.shape[0]}")

Resampled training set size 2062670


In [11]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_resampled, Y_train_resampled)
y_pred = xgb_model.predict(x_test_preprocessed)
print("Accuracy:", accuracy_score(y_test, y_pred))

c:\ccf_project\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [07:41:22] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.9946941215030751


In [12]:
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    257834
           1       0.53      0.85      0.65      1501

    accuracy                           0.99    259335
   macro avg       0.76      0.92      0.82    259335
weighted avg       1.00      0.99      1.00    259335



In [13]:
from sklearn.metrics import average_precision_score
y_proba = xgb_model.predict_proba(x_test_preprocessed)[:, 1]
print(average_precision_score(y_test, y_proba))

0.803481439790529


In [14]:
# Xgboost Stats:-
# The model has a precision of: 0.53 
# Recall is 0.85.
# f1 - score is 0.65
# average_precision_score = 0.8034
# - V1.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_cols = ['amt',
            'lat', 'long', 
            'city_pop', 'merch_lat', 
            'merch_long', 'age',
            'month_sin', 'month_cos']

catagorical_features = ['state', 'category']

# Creating a preprocessing pipelinge
numerical_pipeline = Pipeline(steps =[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

catagorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

preproccessor = ColumnTransformer(
    transformers = [
        ('num', numerical_pipeline, numeric_cols),
        ('cat', catagorical_pipeline, catagorical_features)
    ]
)

pipeline = Pipeline(steps = [
    ('preproccessor', preproccessor),
    ('classifier', xgb_model)
])

pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)
print(accuracy_score(y_test, y_pred))

c:\ccf_project\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [07:41:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0.9969923072473827


In [18]:
from sklearn.metrics import average_precision_score
y_proba = pipeline.predict_proba(x_test)[:, 1]
print(average_precision_score(y_test, y_proba))

0.7197274897530023


In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    257834
           1       0.79      0.66      0.72      1501

    accuracy                           1.00    259335
   macro avg       0.89      0.83      0.86    259335
weighted avg       1.00      1.00      1.00    259335



In [ ]:
# Final Pipeline Stats:-
# The model has a precision of: 0.79
# Recall is 0.72.
# f1 - score is 0.72
# average_precision_score = 0.71
# - Final pipeline before dump

In [ ]:
MODEL_DIR = BASE_DIR/"models"
joblib.dump(pipeline, MODEL_DIR/"XGBoost_fraud_model.pkl")
print("XGBoost model saved: models/XGBoost_fraud_model.pkl")

XGBoost model saved: models/XGBoost_fraud_model.pkl
